In [ ]:
import json, re, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm

RUN_IDS    = [27, 25, 20, 15, 14, 21, 24, 26, 10, 22, 23, 28]
RUNS_DIR   = Path("runs")
OUT_DIR    = Path("presentation_v1")
SHOW_PLOTS = False      

# shared test set
TEST_N    = 10_000         
TEST_SEED = 2026            
N_JOBS    = -1             

T_MAX, NUM_TIME_STEPS = 30.0, 1000
PARAM_LOW   = np.array([-1.0, 0.1, 0.0, 0.0, 0.5, -2.0, -2.0])
PARAM_HIGH  = np.array([ 1.0, 1.0, 0.5, 1.0, 1.5,  2.0,  2.0])
PARAM_NAMES = ["alpha", "beta", "delta", "gamma", "omega", "x0", "v0"]
t_eval = np.linspace(0, T_MAX, NUM_TIME_STEPS)


TOLERANCE      = 0.05   
VALID_TOL      = 0.10   
LYAP_T         = 200.0      
LYAP_THRESHOLD = 0.02       
BOOTSTRAP      = 1000      


C1 = "#1fa08e"   
C2 = "#a01f31"  
C3 = "#8e1fa0"   
C4 = "#31a01f"   


OSCILLATORS = [
    ("single well, regular",           [ 0.5, 0.4, 0.30, 0.50, 1.0, 1.0, 0.0]),
    ("double well, stays in one well", [-0.8, 0.6, 0.40, 0.15, 1.0, 1.2, 0.0]),
    ("double well, chaotic",           [-1.0, 1.0, 0.30, 0.50, 1.2, 1.0, 0.0]),
    ("weakly damped, strongly forced", [ 1.0, 1.0, 0.05, 0.80, 1.4, 0.0, 0.0]),
]

OUT_DIR.mkdir(exist_ok=True)

In [ ]:
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

def param_str(p):
    a, b, d, g, w, x0, v0 = p
    return f"α={a:.2f}  β={b:.2f}  δ={d:.2f}  γ={g:.2f}  ω={w:.2f}  x0={x0:.2f}  v0={v0:.2f}"

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

def finish(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

def star_best(ax, x, y, color, mode="min"):
    x, y = np.asarray(x), np.asarray(y, float)
    ok = ~np.isnan(y)
    if not ok.any():
        return None, None
    i = np.flatnonzero(ok)[np.argmin(y[ok]) if mode == "min" else np.argmax(y[ok])]
    ax.scatter(x[i], y[i], marker="*", s=170, color=color, edgecolor="black", linewidth=0.6, zorder=5)
    return x[i], y[i]

def plot_history_losses(h, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ep = h["epoch"].values
    for key, color, label in [("total", C1, "total"), ("phys", C2, "physics"), ("truth", C3, "ground truth")]:
        if f"train_{key}" in h:
            ax.plot(ep, h[f"train_{key}"], color=color, lw=0.8, alpha=0.4, label=f"{label} (train)")
        if f"val_{key}" in h:
            y = h[f"val_{key}"].values
            m = ~np.isnan(y)
            ax.plot(ep[m], y[m], color=color, lw=1.0, label=f"{label} (validation)")
            star_best(ax, ep, y, color)
    ax.set_yscale("log")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss")
    ax.set_title(f"Run v{run}: loss history (train faded, validation solid, ★ = minimum)")
    ax.legend(fontsize=8, ncol=2)
    finish(fig, path)

def plot_history_l2(h, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ep = h["epoch"].values
    for key, color, label in [("val_rel_l2_median", C1, "median"), ("val_rel_l2_mean", C2, "mean")]:
        if key in h:
            y = 100 * h[key].values
            m = ~np.isnan(y)
            be, bv = star_best(ax, ep, y, color)
            ax.plot(ep[m], y[m], color=color, lw=1.0, label=f"{label} (min {bv:.2f}% at epoch {be})")
    ax.set_yscale("log")
    ax.set_xlabel("epoch"); ax.set_ylabel("relative L2 error [%]")
    ax.set_title(f"Run v{run}: validation relative L2 error during training")
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_history_tol(h, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ep = h["epoch"].values
    y = 100 * h["val_tol_acc"].values
    m = ~np.isnan(y)
    be, bv = star_best(ax, ep, y, C1, mode="max")
    ax.plot(ep[m], y[m], color=C1, lw=1.0, label=f"max {bv:.2f}% at epoch {be}")
    ax.set_ylim(0, 100)
    ax.set_xlabel("epoch"); ax.set_ylabel("%")
    ax.set_title(f"Run v{run}: validation accuracy within ±{TOLERANCE} during training")
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_history_causal(h, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ep = h["epoch"].values
    ax.plot(ep, h["causal_w_mean"], color=C1, lw=1.3, label="mean causal weight w")
    ax.set_ylim(0, 1)
    ax.set_xlabel("epoch"); ax.set_ylabel("w")
    ax2 = ax.twinx()
    ax2.plot(ep, 100 * h["val_rel_l2_median"], color=C2, lw=0.9, label="validation median rel. L2")
    ax2.set_yscale("log"); ax2.set_ylabel("validation median rel. L2 [%]"); ax2.grid(False)
    lines = ax.get_lines() + ax2.get_lines()
    ax.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="center right")
    ax.set_title(f"Run v{run}: causal weight during training")
    finish(fig, path)

def plot_error_vs_time(t, err, Y_true, path, run):
    ae = np.abs(err)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(t, np.median(ae, axis=0),          color=C1, lw=1.5, label="median |error|")
    ax.plot(t, np.quantile(ae, 0.9, axis=0),   color=C2, lw=1.5, label="90th percentile |error|")
    ax.plot(t, np.median(np.abs(Y_true), axis=0), color=C3, lw=1.0, label="median |x| (signal size)")
    ax.axhline(TOLERANCE, color=C4, lw=1.2, ls=":", label=f"tolerance {TOLERANCE}")
    ax.set_xlabel("t [s]"); ax.set_ylabel("|x_network − x_solve_ivp|")
    ax.set_title(f"Run v{run}: error vs time ({len(err)} test trajectories)")
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_error_vs_param(P, rel, i, path, run):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(P[:, i], 100 * rel, s=3, alpha=0.25, color=C1, label=f"test trajectories (n={len(rel)})")
    bins = np.linspace(PARAM_LOW[i], PARAM_HIGH[i], 11)
    idx = np.clip(np.digitize(P[:, i], bins) - 1, 0, 9)
    centers = 0.5 * (bins[1:] + bins[:-1])
    med = [np.median(100 * rel[idx == b]) if np.any(idx == b) else np.nan for b in range(10)]
    ax.plot(centers, med, color=C2, lw=2.2, label="binned median")
    ax.set_yscale("log")
    ax.set_xlabel(PARAM_NAMES[i]); ax.set_ylabel("relative L2 error [%]")
    ax.set_title(f"Run v{run}: relative L2 error vs {PARAM_NAMES[i]}")
    ax.legend(fontsize=8, markerscale=4)
    finish(fig, path)

def plot_error_hist(rel, chaotic, path, run):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    lo = max(100 * rel.min(), 1e-2)
    bins = np.logspace(np.log10(lo), np.log10(100 * rel.max() + 1e-9), 50)
    ax.hist(100 * rel[~chaotic], bins=bins, color=C4, alpha=0.65, label=f"regular (n={(~chaotic).sum()})")
    ax.hist(100 * rel[chaotic],  bins=bins, color=C3, alpha=0.65, label=f"chaotic (n={chaotic.sum()})")
    ax.set_xscale("log")
    ax.set_xlabel("relative L2 error [%]"); ax.set_ylabel("number of trajectories")
    ax.set_title(f"Run v{run}: distribution of the relative L2 error")
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_valid_time(vt, chaotic, path, run):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bins = np.linspace(0, T_MAX, 31)
    ax.hist(vt[~chaotic], bins=bins, color=C4, alpha=0.65, label=f"regular (median {np.median(vt[~chaotic]):.1f} s)" if (~chaotic).any() else "regular")
    ax.hist(vt[chaotic],  bins=bins, color=C3, alpha=0.65, label=f"chaotic (median {np.median(vt[chaotic]):.1f} s)" if chaotic.any() else "chaotic")
    ax.set_xlabel(f"valid prediction time [s]  (until |error| first exceeds {VALID_TOL})")
    ax.set_ylabel("number of trajectories")
    ax.set_title(f"Run v{run}: how long the network stays on the true trajectory ({T_MAX:.0f} s = whole run)")
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_example(t, x_true, x_net, p, rel_i, label, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(t, x_true, color=C2, lw=3.0, label="solve_ivp")
    ax.plot(t, x_net,  color=C1, lw=1.2, label="network")
    ax.set_xlabel("t [s]"); ax.set_ylabel("x")
    ax.set_title(f"Run v{run}: {label} test trajectory (relative L2 {rel_i:.1%})\n{param_str(p)}", fontsize=10)
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_sens_over_time(t, S_true, S_net, group_label, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for S, color, label in [(S_true, C2, "true system (variational equations)"), (S_net, C1, "network (autograd)")]:
        q25, q50, q75 = np.quantile(S, [0.25, 0.5, 0.75], axis=0)
        ax.plot(t, q50, color=color, lw=1.6, label=f"{label}, median")
        ax.fill_between(t, q25, q75, color=color, alpha=0.12, lw=0, label=f"{label}, middle 50% of trajectories")
        ax.plot(t, q25, color=color, lw=0.6, alpha=0.7)
        ax.plot(t, q75, color=color, lw=0.6, alpha=0.7)
    ax.set_yscale("log")
    ax.set_xlabel("t [s]"); ax.set_ylabel("|∂x(t) / ∂(x0, v0)|")
    ax.set_title(f"Run v{run}: sensitivity to initial conditions, {group_label}")
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_sens_scatter(A_true, A_net, chaotic, path, run):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(A_true[~chaotic], A_net[~chaotic], s=5, alpha=0.4, color=C4, label="regular")
    ax.scatter(A_true[chaotic],  A_net[chaotic],  s=5, alpha=0.5, color=C3, label="chaotic")
    lo = max(min(A_true.min(), A_net.min()), 1e-3)
    hi = max(A_true.max(), A_net.max())
    ax.plot([lo, hi], [lo, hi], color=C2, lw=1.2, ls=":", label="network = true")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("true peak amplification  max_t |∂x/∂(x0, v0)|")
    ax.set_ylabel("network peak amplification")
    ax.set_title(f"Run v{run}: how much a small change in (x0, v0) can grow")
    ax.legend(fontsize=8, markerscale=3)
    finish(fig, path)

def plot_sens_example(t, s_true, s_net, p, label, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(t, s_true, color=C2, lw=3.0, label="true system")
    ax.plot(t, s_net,  color=C1, lw=1.2, label="network")
    ax.set_yscale("log")
    ax.set_xlabel("t [s]"); ax.set_ylabel("|∂x(t) / ∂(x0, v0)|")
    ax.set_title(f"Run v{run}: sensitivity, {label} case\n{param_str(p)}", fontsize=10)
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_osc_overlay(t, x_true, x_net, p, name, rel_i, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(t, x_true, color=C2, lw=3.0, label="solve_ivp")
    ax.plot(t, x_net,  color=C1, lw=1.2, label="network")
    ax.set_xlabel("t [s]"); ax.set_ylabel("x")
    ax.set_title(f"Run v{run}: {name}, network vs solve_ivp (relative L2 {rel_i:.1%})\n{param_str(p)}", fontsize=10)
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_osc_single(t, x, color, who, p, name, path, run):
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot(t, x, color=color, lw=1.4, label=who)
    ax.set_xlabel("t [s]"); ax.set_ylabel("x")
    ax.set_title(f"Run v{run}: {name}, {who}\n{param_str(p)}", fontsize=10)
    ax.legend(fontsize=8)
    finish(fig, path)

def plot_osc_phase(x_net, v_net, x_true, v_true, p, name, field, path, run, n_grid=21):
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    xa = np.concatenate([x_net, x_true]); va = np.concatenate([v_net, v_true])
    pad_x, pad_v = 0.12 * (np.ptp(xa) or 1.0), 0.12 * (np.ptp(va) or 1.0)
    x_lim = (xa.min() - pad_x, xa.max() + pad_x)
    v_lim = (va.min() - pad_v, va.max() + pad_v)
    if field is not None:
        draw_field(ax, field, x_lim, v_lim, n_grid)
    ax.plot(x_true, v_true, color=C2, lw=2.6, label="solve_ivp trajectory")
    ax.plot(x_net,  v_net,  color=C1, lw=1.1, label="network trajectory")
    ax.scatter(x_net[0], v_net[0], color=C4, s=50, zorder=5, label="start (x0, v0)")
    ax.set_xlim(x_lim); ax.set_ylim(v_lim)
    ax.set_xlabel("x"); ax.set_ylabel("v = dx/dt")
    ax.set_title(f"Run v{run}: phase portrait, {name}\n{param_str(p)}", fontsize=10)
    ax.legend(fontsize=8, loc="upper right", framealpha=0.9)
    finish(fig, path)

def draw_field(ax, field, x_lim, v_lim, n_grid):
    gx = np.linspace(max(x_lim[0], PARAM_LOW[5]), min(x_lim[1], PARAM_HIGH[5]), n_grid)
    gv = np.linspace(max(v_lim[0], PARAM_LOW[6]), min(v_lim[1], PARAM_HIGH[6]), n_grid)
    X, V = np.meshgrid(gx, gv)
    DX, DV = field(X.ravel(), V.ravel())
    sx, sv = x_lim[1] - x_lim[0], v_lim[1] - v_lim[0]
    ux, uv = DX / sx, DV / sv                           
    norm = np.hypot(ux, uv); norm[norm == 0] = 1.0
    L = 0.35 / n_grid                                   
    ax.quiver(X.ravel(), V.ravel(), ux / norm * L * sx, uv / norm * L * sv, angles="xy", scale_units="xy",
              scale=1, color=C3, alpha=0.7, width=0.0018, headwidth=4, headlength=4.5, headaxislength=4,
              label="network direction field (dx/dt, dv/dt) at t = 0")

def plot_summary(labels, series, ylabel, title, path, log=False, scale=1.0):
    fig, ax = plt.subplots(figsize=(11, 4.8))
    x = np.arange(len(labels))
    for k, (name, color, est, lo, hi) in enumerate(series):
        est, lo, hi = (scale * np.asarray(a, float) for a in (est, lo, hi))
        xx = x + (k - (len(series) - 1) / 2) * 0.22
        ax.errorbar(xx, est, yerr=[est - lo, hi - est], fmt="o", color=color, capsize=3, ms=5, label=name)
    ax.set_xticks(x, labels)
    if log:
        ax.set_yscale("log")
    ax.set_xlabel("run"); ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    finish(fig, path)

In [ ]:
def duffing_tangent(t, s, alpha, beta, delta, gamma, omega):
    x, v, sx, sv, rx, rv = s
    k = alpha + 3 * beta * x**2                       
    return [v, gamma * np.cos(omega * t) - delta * v - alpha * x - beta * x**3,
            sv, -delta * sv - k * sx,                  
            rv, -delta * rv - k * rx]                  

LYAP_WIN = 200                                       
T_ALL = np.concatenate([t_eval,
                        np.linspace(LYAP_T / 2 - 10, LYAP_T / 2, LYAP_WIN),
                        np.linspace(LYAP_T - 10, LYAP_T, LYAP_WIN)])

def solve_one(p):
    a, b, d, g, w, x0, v0 = p
    sol = solve_ivp(duffing_tangent, [0, LYAP_T], [x0, v0, 1.0, 0.0, 0.0, 1.0], args=(a, b, d, g, w),
                    t_eval=T_ALL, method="DOP853", rtol=1e-8, atol=1e-10)
    if sol.y.shape[1] != T_ALL.size:                 # solver failure
        return None
    y, n = sol.y, NUM_TIME_STEPS
    tangent = np.hypot(y[2, n:], y[3, n:])           
    r1 = np.sqrt(np.mean(tangent[:LYAP_WIN] ** 2))
    r2 = np.sqrt(np.mean(tangent[LYAP_WIN:] ** 2))
    lam = (np.log(r2) - np.log(r1)) / (LYAP_T / 2)    
    return y[0, :n], y[1, :n], np.hypot(y[2, :n], y[4, :n]), lam

test_path = OUT_DIR / f"test_set_{TEST_N // 1000}k.npz"
signature = json.dumps({"n": TEST_N, "seed": TEST_SEED, "t_max": T_MAX, "steps": NUM_TIME_STEPS,
                        "low": PARAM_LOW.tolist(), "high": PARAM_HIGH.tolist(), "lyap_t": LYAP_T})

if test_path.exists() and str(np.load(test_path)["signature"]) == signature:
    data = np.load(test_path)
    P_test, X_test, V_test, S_test, LAM_test = (data[k] for k in ["P", "X", "V", "S", "LAM"])
    print(f"Loaded cached test set: {len(P_test)} trajectories")
else:
    rng = np.random.default_rng(TEST_SEED)
    P_all = rng.uniform(PARAM_LOW, PARAM_HIGH, size=(TEST_N, 7))
    start = time.time()
    try:
        from joblib import Parallel, delayed
        results = Parallel(n_jobs=N_JOBS, batch_size=32)(
            delayed(solve_one)(p) for p in tqdm(P_all, desc="Solving test set (parallel)"))
    except ImportError:
        print("joblib not installed: solving on one core (pip install joblib to use all cores)")
        results = [solve_one(p) for p in tqdm(P_all, desc="Solving test set")]
    ok = np.array([r is not None for r in results])
    if not ok.all():
        print(f"{(~ok).sum()} trajectories failed to solve and were dropped")
    good = [r for r in results if r is not None]
    P_test   = P_all[ok]
    X_test   = np.stack([r[0] for r in good]).astype(np.float32)
    V_test   = np.stack([r[1] for r in good]).astype(np.float32)
    S_test   = np.stack([r[2] for r in good]).astype(np.float32)
    LAM_test = np.array([r[3] for r in good])
    np.savez_compressed(test_path, P=P_test, X=X_test, V=V_test, S=S_test, LAM=LAM_test, signature=signature)
    print(f"Solved {len(P_test)} trajectories in {(time.time() - start) / 60:.1f} min, saved to {test_path}")

chaotic_test = LAM_test > LYAP_THRESHOLD
info = [f"Test set: {len(P_test)} trajectories, seed {TEST_SEED}, uniform over the training parameter ranges",
        f"t = 0 ... {T_MAX:g} s, {NUM_TIME_STEPS} time points; solve_ivp DOP853, rtol 1e-8, atol 1e-10",
        f"Chaotic (Lyapunov exponent over {LYAP_T:g} s > {LYAP_THRESHOLD} 1/s): {chaotic_test.sum()} "
        f"({chaotic_test.mean():.1%});  regular: {(~chaotic_test).sum()}",
        f"alpha < 0 (double well): {(P_test[:, 0] < 0).sum()};  of these chaotic: "
        f"{chaotic_test[P_test[:, 0] < 0].mean():.1%}",
        f"Lyapunov exponent: median {np.median(LAM_test):+.3f} 1/s, max {LAM_test.max():+.3f} 1/s"]
(OUT_DIR / "test_set_info.txt").write_text("\n".join(info), encoding="utf-8")
print("\n".join(info))

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

ACTIVATIONS = {"GELU": nn.GELU, "SiLU": nn.SiLU, "Mish": nn.Mish, "Tanh": nn.Tanh,
               "Softplus": nn.Softplus, "Sigmoid": nn.Sigmoid, "ReLU": nn.ReLU, "LeakyReLU": nn.LeakyReLU}

class DuffingNet(nn.Module):
    def __init__(self, widths, activation, normalize=False, hard_ic=False, tau_c=1.0):
        super().__init__()
        self.normalize, self.hard_ic, self.tau_c = normalize, hard_ic, tau_c
        lo = torch.tensor(list(PARAM_LOW)  + [0.0], dtype=torch.float32)
        hi = torch.tensor(list(PARAM_HIGH) + [1.0], dtype=torch.float32)
        self.register_buffer("in_mid",  (hi + lo) / 2, persistent=False)
        self.register_buffer("in_half", (hi - lo) / 2, persistent=False)
        layers = []
        for i in range(len(widths) - 2):
            layers += [nn.Linear(widths[i], widths[i + 1]), activation()]
        layers.append(nn.Linear(widths[-2], widths[-1]))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        raw = x
        if self.normalize:
            x = (x - self.in_mid) / self.in_half
        out = self.net(x)
        if not self.hard_ic:
            return out
        x0, v0 = raw[:, 5:6], raw[:, 6:7]
        t = raw[:, 7:8] * T_MAX
        s = 1 - torch.exp(-t / self.tau_c)
        return x0 + v0 * self.tau_c * s + s**2 * out

T_STEPS = NUM_TIME_STEPS
t_grid = torch.tensor(t_eval / T_MAX, dtype=torch.float32, device=device)

def load_run(run):
    run_dir = RUNS_DIR / f"v{run}"
    cfg = json.loads((run_dir / "config.json").read_text())
    if float(cfg.get("t_max", T_MAX)) != T_MAX or int(cfg.get("num_time_steps", NUM_TIME_STEPS)) != NUM_TIME_STEPS:
        raise ValueError(f"run v{run} used t_max={cfg.get('t_max')}, num_time_steps={cfg.get('num_time_steps')}")
    if not (np.allclose(cfg.get("param_low", PARAM_LOW), PARAM_LOW) and np.allclose(cfg.get("param_high", PARAM_HIGH), PARAM_HIGH)):
        raise ValueError(f"run v{run} used different parameter ranges")
    widths = [int(w) for w in cfg["architecture"].split("-")]
    act = ACTIVATIONS[str(cfg.get("activation", "GELU")).split(",")[0]]
    model = DuffingNet(widths, act,
                       normalize=bool(cfg.get("normalize_inputs") or False),
                       hard_ic=bool(cfg.get("hard_ic") or False),
                       tau_c=float(cfg.get("tau_c") or 1.0)).to(device)
    state = torch.load(run_dir / f"duffing_net_best_v{run}.pt", map_location=device)
    if isinstance(state, dict) and not any(torch.is_tensor(v) for v in state.values()):
        for key in ("model_state_dict", "state_dict", "model"):  
            if key in state:
                state = state[key]
                break
    with torch.no_grad():                                            
        for name in ("in_mid", "in_half"):
            if name in state and state[name].shape == getattr(model, name).shape:
                getattr(model, name).copy_(state[name])
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        raise ValueError(f"checkpoint of v{run} does not match the rebuilt network, missing: {missing[:4]}")
    extra = [k for k in unexpected if k not in ("in_mid", "in_half")]
    if extra:
        print(f"v{run}: ignored extra checkpoint entries {extra}")
    model.eval()
    return model, cfg, run_dir

def _inputs(p):
    B = p.shape[0]
    return torch.cat([p.unsqueeze(1).expand(B, T_STEPS, 7),
                      t_grid.view(1, T_STEPS, 1).expand(B, T_STEPS, 1)], dim=-1).reshape(B * T_STEPS, 8)

@torch.no_grad()
def predict(model, P, chunk=100):
    out = np.empty((len(P), T_STEPS), np.float32)
    for i in range(0, len(P), chunk):
        p = torch.tensor(P[i:i + chunk], dtype=torch.float32, device=device)
        out[i:i + len(p)] = model(_inputs(p)).view(len(p), T_STEPS).cpu().numpy()
    return out

def net_sensitivity(model, P, chunk=50):
    S = np.empty((len(P), T_STEPS), np.float32)
    for i in range(0, len(P), chunk):
        p = torch.tensor(P[i:i + chunk], dtype=torch.float32, device=device)
        inp = _inputs(p).clone().requires_grad_(True)
        out = model(inp)
        g = torch.autograd.grad(out.sum(), inp)[0]           
        S[i:i + len(p)] = torch.hypot(g[:, 5], g[:, 6]).view(len(p), T_STEPS).detach().cpu().numpy()
    return S

def net_derivatives(model, p):
    tt = t_grid.view(-1, 1).clone().requires_grad_(True)
    pp = torch.tensor(p, dtype=torch.float32, device=device).view(1, 7).expand(T_STEPS, 7)
    x = model(torch.cat([pp, tt], dim=1))
    dx = torch.autograd.grad(x, tt, torch.ones_like(x), create_graph=True)[0] / T_MAX
    ddx = torch.autograd.grad(dx, tt, torch.ones_like(dx))[0] / T_MAX
    return (x.detach().cpu().numpy().ravel(), dx.detach().cpu().numpy().ravel(),
            ddx.detach().cpu().numpy().ravel())

def net_field(model, p, X, V):
    n = len(X)
    pp = torch.tensor(p[:5], dtype=torch.float32, device=device).view(1, 5).expand(n, 5)
    ic = torch.tensor(np.stack([X, V], axis=1), dtype=torch.float32, device=device)
    t = torch.zeros(n, 1, device=device, requires_grad=True)          # seconds
    x = model(torch.cat([pp, ic, t / T_MAX], dim=1))
    dx = torch.autograd.grad(x, t, torch.ones_like(x), create_graph=True)[0]
    ddx = torch.autograd.grad(dx, t, torch.ones_like(dx))[0]
    return dx.detach().cpu().numpy().ravel(), ddx.detach().cpu().numpy().ravel()

def free_memory():
    if device.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
def p90(a, axis=None):
    return np.percentile(a, 90, axis=axis)

def boot(values, fn, seed=0):
    values = np.asarray(values, float)
    n = len(values)
    if n == 0:
        return (np.nan, np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    res = np.empty(BOOTSTRAP)
    for s in range(0, BOOTSTRAP, 100):
        k = min(100, BOOTSTRAP - s)
        res[s:s + k] = fn(values[rng.integers(0, n, size=(k, n))], axis=1)
    return (float(fn(values)), float(res.std(ddof=1)), float(np.percentile(res, 2.5)), float(np.percentile(res, 97.5)))

def fmt_pct(s):
    est, se, lo, hi = s
    return "n/a" if np.isnan(est) else f"{100*est:6.2f}% ± {100*se:.2f}%   (95% CI {100*lo:.2f} – {100*hi:.2f}%)"

def fmt_sec(s):
    est, se, lo, hi = s
    return "n/a" if np.isnan(est) else f"{est:6.2f} s ± {se:.2f} s   (95% CI {lo:.2f} – {hi:.2f} s)"

def fmt_ratio(s):
    est, se, lo, hi = s
    return "n/a" if np.isnan(est) else f"{10**est:.3g}x   (95% CI {10**lo:.3g} – {10**hi:.3g}x)"

def valid_time(err):
    exceed = np.abs(err) > VALID_TOL
    first = np.argmax(exceed, axis=1)
    return np.where(exceed.any(axis=1), t_eval[first], T_MAX)

def group_stats(rel, tolf, vt):
    return {"n": len(rel), "median": boot(rel, np.median), "mean": boot(rel, np.mean), "p90": boot(rel, p90),
            "tol": boot(tolf, np.mean), "valid": boot(vt, np.median)}

OSC_TRUE = [solve_one(np.array(p, float)) for _, p in OSCILLATORS]

def evaluate_run(run):
    t_start = time.time()
    model, cfg, run_dir = load_run(run)
    out = OUT_DIR / f"run{run}"
    out.mkdir(parents=True, exist_ok=True)
    N = len(P_test)

    # ---------- predictions on the shared test set ----------
    Y_net = predict(model, P_test)
    err   = Y_net - X_test
    rel   = np.linalg.norm(err, axis=1) / (np.linalg.norm(X_test, axis=1) + 1e-8)
    tolf  = np.mean(np.abs(err) < TOLERANCE, axis=1)          # per-trajectory tolerance accuracy
    vt    = valid_time(err)

    groups = {"all": np.ones(N, bool),
              f"chaotic (Lyapunov exp. > {LYAP_THRESHOLD} 1/s)": chaotic_test,
              "regular": ~chaotic_test,
              "alpha < 0 (double well)": P_test[:, 0] < 0,
              "alpha >= 0 (single well)": P_test[:, 0] >= 0}
    stats = {g: group_stats(rel[m], tolf[m], vt[m]) for g, m in groups.items()}

    S_net  = net_sensitivity(model, P_test)
    A_true = S_test.max(axis=1)
    A_net  = S_net.max(axis=1)
    log_ratio = np.log10(A_net + 1e-12) - np.log10(A_true + 1e-12)
    sens = {}
    for g, m in [("chaotic", chaotic_test), ("regular", ~chaotic_test)]:
        sens[g] = {"n": int(m.sum()),
                   "true": boot(A_true[m], np.median), "net": boot(A_net[m], np.median),
                   "ratio": boot(log_ratio[m], np.median),
                   "within2": float(np.mean(np.abs(log_ratio[m]) < np.log10(2))) if m.any() else np.nan}

    hist_best = {}
    causal_on = bool(cfg.get("causal") or False)
    hist_path = run_dir / "history.csv"
    if hist_path.exists():
        h = pd.read_csv(hist_path)
        if "epoch" not in h.columns:
            h["epoch"] = np.arange(1, len(h) + 1)
        plot_history_losses(h, out / "history_losses.png", run)
        plot_history_l2(h, out / "history_l2_median_mean.png", run)
        plot_history_tol(h, out / "history_tol_accuracy.png", run)
        if causal_on and "causal_w_mean" in h.columns and h["causal_w_mean"].notna().any():
            plot_history_causal(h, out / "history_causal_weight.png", run)
        i_best = int(np.nanargmin(h["val_rel_l2_median"].values))
        hist_best = {"epoch": int(h["epoch"].iloc[i_best]), "median": float(h["val_rel_l2_median"].iloc[i_best]),
                     "epochs": int(h["epoch"].iloc[-1])}

    plot_error_vs_time(t_eval, err, X_test, out / "error_vs_time.png", run)
    for i, name in enumerate(PARAM_NAMES):
        plot_error_vs_param(P_test, rel, i, out / f"error_vs_{name}.png", run)
    plot_error_hist(rel, chaotic_test, out / "error_distribution.png", run)
    plot_valid_time(vt, chaotic_test, out / "valid_time_distribution.png", run)

    order = np.argsort(rel)
    picks = [("best", order[0]), ("25th percentile", order[N // 4]), ("median", order[N // 2]),
             ("75th percentile", order[3 * N // 4]), ("90th percentile", order[int(0.9 * N)]), ("worst", order[-1])]
    for k, (label, i) in enumerate(picks, 1):
        plot_example(t_eval, X_test[i], Y_net[i], P_test[i], rel[i], label, out / f"example_{k}_{slug(label)}.png", run)

    for g, m in [("chaotic", chaotic_test), ("regular", ~chaotic_test)]:
        if m.any():
            plot_sens_over_time(t_eval, S_test[m], S_net[m], f"{g} trajectories (n={m.sum()})",
                                out / f"sensitivity_over_time_{g}.png", run)
    plot_sens_scatter(A_true, A_net, chaotic_test, out / "sensitivity_peak_scatter.png", run)
    order_A = np.argsort(A_true)
    sens_picks = [("weak", order_A[N // 4]), ("median", order_A[N // 2]),
                  ("strong (90th percentile)", order_A[int(0.9 * N)]), ("strongest", order_A[-1])]
    for k, (label, i) in enumerate(sens_picks, 1):
        plot_sens_example(t_eval, S_test[i], S_net[i], P_test[i], label,
                          out / f"sensitivity_example_{k}_{slug(label)}.png", run)

    osc_rows = []
    for k, ((name, p), true) in enumerate(zip(OSCILLATORS, OSC_TRUE), 1):
        x_true, v_true, _, lam = true
        x_net, v_net, a_net = net_derivatives(model, p)
        e = x_net - x_true
        rel_k = float(np.linalg.norm(e) / (np.linalg.norm(x_true) + 1e-8))
        vt_k = float(valid_time(e[None])[0])
        plot_osc_overlay(t_eval, x_true, x_net, p, name, rel_k, out / f"osc{k}_overlay.png", run)
        plot_osc_single(t_eval, x_net,  C1, "network",   p, name, out / f"osc{k}_network.png", run)
        plot_osc_single(t_eval, x_true, C2, "solve_ivp", p, name, out / f"osc{k}_solve_ivp.png", run)
        plot_osc_phase(x_net, v_net, x_true, v_true, p, name, None,
                       out / f"osc{k}_phase.png", run)
        plot_osc_phase(x_net, v_net, x_true, v_true, p, name, lambda X, V: net_field(model, p, X, V),
                       out / f"osc{k}_phase_arrows.png", run)
        osc_rows.append((k, name, p, lam, rel_k, vt_k))
    L = []
    L.append(f"RUN v{run}: evaluation on {N} fresh test trajectories (seed {TEST_SEED})")
    L.append("=" * 78)
    L.append(f"note:          {cfg.get('note', '')}")
    L.append(f"architecture:  {cfg.get('architecture')}   activation: {cfg.get('activation')}   "
             f"weights: {cfg.get('n_params')}")
    L.append(f"training data: {cfg.get('num_samples')} samples (train {cfg.get('train_size')}), "
             f"batch {cfg.get('batch_size')}, XI {cfg.get('xi')}, N_COL {cfg.get('n_col')}")
    L.append(f"schedule:      lr {cfg.get('lr')} -> {cfg.get('lr_min')}, cosine {cfg.get('cosine_epochs')}, "
             f"epochs {cfg.get('epochs_planned')}, weight decay {cfg.get('weight_decay')}")
    L.append(f"extras:        causal {cfg.get('causal', False)} (eps {cfg.get('causal_eps')}), "
             f"normalized inputs {cfg.get('normalize_inputs', False)}, hard IC {cfg.get('hard_ic', False)}")
    if hist_best:
        L.append(f"training:      best validation median {100*hist_best['median']:.2f}% at epoch "
                 f"{hist_best['epoch']} of {hist_best['epochs']}  (checkpoint evaluated here)")
        if np.median(rel) > 2 * hist_best["median"]:
            L.append("WARNING:       test median is more than twice the validation median - "
                     "check that the network was rebuilt exactly as in training")
    L.append("")
    L.append("Uncertainties: '±' is the bootstrap standard error over test trajectories "
             f"({BOOTSTRAP} resamples), the bracket the 95% bootstrap interval.")
    L.append("They cover the finite test set only, not run-to-run variation from training.")
    L.append("")
    for g, s in stats.items():
        L.append(f"--- {g}: n = {s['n']}")
        L.append(f"  relative L2 median          {fmt_pct(s['median'])}")
        L.append(f"  relative L2 mean            {fmt_pct(s['mean'])}")
        L.append(f"  relative L2 90th percentile {fmt_pct(s['p90'])}")
        L.append(f"  within ±{TOLERANCE} accuracy      {fmt_pct(s['tol'])}")
        L.append(f"  valid prediction time (median, |error| < {VALID_TOL}): {fmt_sec(s['valid'])}")
        L.append("")
    L.append("--- sensitivity to initial conditions: peak of |dx(t)/d(x0, v0)| over 0-30 s")
    for g, s in sens.items():
        L.append(f"  {g} (n = {s['n']}):")
        L.append(f"    median peak, true system   {s['true'][0]:.4g}   (95% CI {s['true'][2]:.4g} – {s['true'][3]:.4g})")
        L.append(f"    median peak, network       {s['net'][0]:.4g}   (95% CI {s['net'][2]:.4g} – {s['net'][3]:.4g})")
        L.append(f"    median ratio network/true  {fmt_ratio(s['ratio'])}")
        L.append(f"    network within a factor 2 of the true peak: {s['within2']:.1%}")
    L.append("")
    L.append("--- fixed oscillators")
    for k, name, p, lam, rel_k, vt_k in osc_rows:
        L.append(f"  osc{k}: {name}")
        L.append(f"    {param_str(p)}")
        L.append(f"    Lyapunov exponent {lam:+.3f} 1/s ({'chaotic' if lam > LYAP_THRESHOLD else 'regular'}), "
                 f"relative L2 {rel_k:.2%}, valid prediction time {vt_k:.1f} s")
    L.append("")
    L.append(f"evaluation time: {time.time() - t_start:.0f} s")
    (out / "results.txt").write_text("\n".join(L), encoding="utf-8")

    del model
    free_memory()

    row = {"run": run, "architecture": cfg.get("architecture"), "activation": cfg.get("activation"),
           "note": cfg.get("note", ""), "val_median_training": hist_best.get("median", np.nan)}
    for g_key, g in [("all", "all"), ("chaotic", list(groups)[1]), ("regular", "regular")]:
        for metric in ["median", "mean", "tol", "valid"]:
            row[f"{g_key}_{metric}"] = stats[g][metric]
    row["sens_ratio_chaotic"] = sens["chaotic"]["ratio"]
    return row

In [ ]:
summary_rows, failed = [], []
errors_path = OUT_DIR / "errors.txt"
if errors_path.exists():
    errors_path.unlink()

for run in tqdm(RUN_IDS, desc="Runs"):
    try:
        t0 = time.time()
        summary_rows.append(evaluate_run(run))
        tqdm.write(f"v{run}: done in {time.time() - t0:.0f} s -> {OUT_DIR / f'run{run}'}")
    except Exception:
        failed.append(run)
        with open(errors_path, "a", encoding="utf-8") as f:
            f.write(f"=== run v{run} ===\n{traceback.format_exc()}\n")
        tqdm.write(f"v{run}: FAILED, see {errors_path}")
    finally:
        plt.close("all")
        free_memory()

print(f"\nEvaluated {len(summary_rows)} of {len(RUN_IDS)} runs" + (f"; failed: {failed}" if failed else ""))

In [ ]:
sum_dir = OUT_DIR / "summary"
sum_dir.mkdir(exist_ok=True)

if summary_rows:
    labels = [f"v{r['run']}" for r in summary_rows]
    col = lambda key, j: [r[key][j] for r in summary_rows]

    plot_summary(labels, [("median", C1, col("all_median", 0), col("all_median", 2), col("all_median", 3)),
                          ("mean",   C2, col("all_mean", 0),   col("all_mean", 2),   col("all_mean", 3))],
                 "relative L2 error [%]", f"Relative L2 error on {len(P_test)} test trajectories (95% CI)",
                 sum_dir / "summary_l2_all.png", scale=100)
    plot_summary(labels, [("chaotic", C3, col("chaotic_mean", 0), col("chaotic_mean", 2), col("chaotic_mean", 3)),
                          ("regular", C4, col("regular_mean", 0), col("regular_mean", 2), col("regular_mean", 3))],
                 "mean relative L2 error [%]", "Mean relative L2 error, chaotic vs regular trajectories (95% CI)",
                 sum_dir / "summary_l2_mean_by_regime.png", log=True, scale=100)
    plot_summary(labels, [("chaotic", C3, col("chaotic_median", 0), col("chaotic_median", 2), col("chaotic_median", 3)),
                          ("regular", C4, col("regular_median", 0), col("regular_median", 2), col("regular_median", 3))],
                 "median relative L2 error [%]", "Median relative L2 error, chaotic vs regular trajectories (95% CI)",
                 sum_dir / "summary_l2_median_by_regime.png", log=True, scale=100)
    plot_summary(labels, [("chaotic", C3, col("chaotic_valid", 0), col("chaotic_valid", 2), col("chaotic_valid", 3)),
                          ("regular", C4, col("regular_valid", 0), col("regular_valid", 2), col("regular_valid", 3))],
                 "valid prediction time [s], median", f"Valid prediction time (|error| < {VALID_TOL}), median with 95% CI",
                 sum_dir / "summary_valid_time_by_regime.png")
    plot_summary(labels, [("all", C1, col("all_tol", 0), col("all_tol", 2), col("all_tol", 3))],
                 f"points within ±{TOLERANCE} [%]", f"Tolerance accuracy on {len(P_test)} test trajectories (95% CI)",
                 sum_dir / "summary_tol_accuracy.png", scale=100)

    pct = lambda s: f"{100*s[0]:.2f} ± {100*s[1]:.2f}"
    lines = [f"Comparison of runs on the same {len(P_test)} test trajectories (seed {TEST_SEED})",
             f"chaotic = Lyapunov exponent > {LYAP_THRESHOLD} 1/s: {chaotic_test.sum()} trajectories "
             f"({chaotic_test.mean():.1%}); regular: {(~chaotic_test).sum()}",
             "values in %, '±' = bootstrap standard error over test trajectories; valid time in s (median)",
             "",
             f"{'run':>5} {'val med (train)':>15} {'median':>14} {'mean':>14} {'tol acc':>14} "
             f"{'chaotic mean':>14} {'regular mean':>14} {'chaotic med':>14} {'regular med':>14} "
             f"{'valid chaot':>11} {'valid reg':>10} {'sens net/true chaotic':>22}  architecture / activation / note"]
    for r in sorted(summary_rows, key=lambda r: r["all_median"][0]):
        lines.append(f"{'v' + str(r['run']):>5} {100*r['val_median_training']:>14.2f}% "
                     f"{pct(r['all_median']):>14} {pct(r['all_mean']):>14} {pct(r['all_tol']):>14} "
                     f"{pct(r['chaotic_mean']):>14} {pct(r['regular_mean']):>14} "
                     f"{pct(r['chaotic_median']):>14} {pct(r['regular_median']):>14} "
                     f"{r['chaotic_valid'][0]:>10.1f}s {r['regular_valid'][0]:>9.1f}s "
                     f"{10**r['sens_ratio_chaotic'][0]:>21.3g}x  {r['architecture']} / {r['activation']} / {r['note']}")
    if failed:
        lines += ["", f"failed runs (see errors.txt): {failed}"]
    (sum_dir / "summary.txt").write_text("\n".join(lines), encoding="utf-8")
    print("\n".join(lines))
else:
    print("No run was evaluated successfully; see errors.txt")